In [2]:
import numpy as np
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1, 784).astype(np.float32) / 255.0
X_test = X_test.reshape(-1, 784).astype(np.float32) / 255.0

#Encoding

def one_hot(y, num_classes=10):
    result = np.zeros((len(y), num_classes))
    result[np.arange(len(y)), y] = 1
    return result

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

# Activations

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Loss

def cross_entropy(y_true, y_pred):

    m = y_true.shape[0]

    return -np.sum(
        y_true * np.log(y_pred + 1e-9)
    ) / m

# Accuracy

def accuracy(predictions, labels):

    predicted_classes = np.argmax(
        predictions,
        axis=1
    )

    return np.mean(predicted_classes == labels)

# MLP

class MLP:

    def __init__(self):

        # He Initialization

        self.W1 = np.random.randn(
            784, 128
        ) * np.sqrt(2 / 784)

        self.b1 = np.zeros((1, 128))

        self.W2 = np.random.randn(
            128, 64
        ) * np.sqrt(2 / 128)

        self.b2 = np.zeros((1, 64))

        self.W3 = np.random.randn(
            64, 10
        ) * np.sqrt(2 / 64)

        self.b3 = np.zeros((1, 10))


    def forward(self, X):

        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)

        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = relu(self.Z2)

        self.Z3 = self.A2 @ self.W3 + self.b3

        self.output = softmax(self.Z3)

        return self.output


    def backward(self, X, Y, learning_rate):

        m = X.shape[0]

        # Output layer

        dZ3 = self.output - Y

        dW3 = (self.A2.T @ dZ3) / m
        db3 = np.sum(
            dZ3,
            axis=0,
            keepdims=True
        ) / m

        # Hidden Layer 2

        dA2 = dZ3 @ self.W3.T

        dZ2 = dA2 * relu_derivative(
            self.Z2
        )

        dW2 = (self.A1.T @ dZ2) / m

        db2 = np.sum(
            dZ2,
            axis=0,
            keepdims=True
        ) / m

        # Hidden Layer 1

        dA1 = dZ2 @ self.W2.T

        dZ1 = dA1 * relu_derivative(
            self.Z1
        )

        dW1 = (X.T @ dZ1) / m

        db1 = np.sum(
            dZ1,
            axis=0,
            keepdims=True
        ) / m

        # Gradient Descent

        self.W3 -= learning_rate * dW3
        self.b3 -= learning_rate * db3

        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2

        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1

# Training

model = MLP()

epochs = 15
batch_size = 128
learning_rate = 0.01

num_samples = X_train.shape[0]

for epoch in range(epochs):

    permutation = np.random.permutation(
        num_samples
    )

    X_train_shuffled = X_train[
        permutation
    ]

    Y_train_shuffled = Y_train[
        permutation
    ]

    for i in range(
        0,
        num_samples,
        batch_size
    ):

        X_batch = X_train_shuffled[
            i:i+batch_size
        ]

        Y_batch = Y_train_shuffled[
            i:i+batch_size
        ]

        model.forward(X_batch)

        model.backward(
            X_batch,
            Y_batch,
            learning_rate
        )

    # Training Metrics

    train_preds = model.forward(
        X_train[:5000]
    )

    train_loss = cross_entropy(
        Y_train[:5000],
        train_preds
    )

    train_acc = accuracy(
        train_preds,
        y_train[:5000]
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_acc:.4f}"
    )

# =====================================================
# Testing
# =====================================================

test_preds = model.forward(X_test)

test_loss = cross_entropy(
    Y_test,
    test_preds
)

test_acc = accuracy(
    test_preds,
    y_test
)

print("\n======================")
print("FINAL RESULTS")
print("======================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

ModuleNotFoundError: No module named 'tensorflow'